# Modeling Scaled Watch Data with RF

In [16]:
import sys
import os
sys.path.append(os.path.abspath('..'))

In [17]:
import sys
import os
sys.path.append(os.path.abspath('..'))
from venus_ml import VenusDataset
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

### Test Parameters

In [18]:
run1 = '00'
run2 = '09'
target = ["fcv1_i_mean"]
random_state = 88
rf_args = {
    "n_estimators":100, 
    "max_features": "log2",
    "random_state": random_state
}

## Dataset

In [19]:
def get_features(names, mean=True, std=False):
    out_list = []
    for name in names:
        if mean:
            out_list.append(name + "_mean")
        if std:
            out_list.append(name + "_std")
    return out_list

In [20]:
feature_groups = {
    "S": ["inj_mbar", "bias_v", "bias_i", "extraction_i"],
    "M": ["inj_mbar", "bias_v", "bias_i", "extraction_i", "ext_mbar", "k18_fw", "k18_ref", "g28_fw", "puller_i"],
    "L": ["inj_mbar", "bias_v", "bias_i", "extraction_i", "ext_mbar", "k18_fw", "k18_ref", "g28_fw", "puller_i", "inj_i", "ext_i", "mid_i", "sext_i", "x_ray_source", "x_ray_exit"],
}

data_set_config = {
    "file_path": "../data/watch_data.parquet",
    "input_columns": [],
    "output_columns": target,
    "run_selection": (run1, run2),
    # "transforms": [generate_smoother(2)]
}
datasets = {}

for feature_group in feature_groups.keys():
    data_set_config["input_columns"] = get_features(feature_groups[feature_group], std=False)
    datasets[feature_group] = VenusDataset(**data_set_config)
    
for dataset in datasets.values():
    x, y = dataset.to_numpy()
    print(dataset.df['run_id'].unique())
    print(x.shape, y.shape)

['00' '09']
(3173, 4) (3173, 1)
['00' '09']
(3173, 9) (3173, 1)
['00' '09']
(3173, 15) (3173, 1)


### Data Split Helper Functions

In [21]:
def one_run(data):
    return data.get_run_splits((run1,))  # train_x, train_y, validation_x, validation_y

def both_runs(data):
    return data.get_run_splits((run1, run2)) # train_x, train_y, validation_x, validation_y

def between_runs(data):
    data = data.get_runs((run1,run2))
    return data[run1][0], data[run1][1], data[run2][0], data[run2][1] # train_x, train_y, validation_x, validation_y

def all_splits(data):
    return {
        "one_run": one_run(data),
        "both_run": both_runs(data),
        "between_runs": between_runs(data)
    }

## Comparison RF vs. ExtraTrees

In [22]:
for size, dataset in datasets.items():
    for run, data in all_splits(dataset).items():
        train_x, train_y, test_x, test_y = data
        train_y, test_y = train_y.ravel(), test_y.ravel()
        rf = RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=random_state)
        ext = ExtraTreesRegressor(n_estimators=200, n_jobs=-1, random_state=random_state)
        rf.fit(train_x, train_y)
        ext.fit(train_x, train_y)
        
        rf_pred = rf.predict(test_x)
        ext_pred = ext.predict(test_x)
        
        rf_err = format(mean_squared_error(test_y, rf_pred), 'e')
        ext_err = format(mean_squared_error(test_y, ext_pred), 'e')
    
        print(f"Dataset {size} | Split: {run} | RF:{rf_err} | EXT: {ext_err}")
    print("\n")
        

AttributeError: 'VenusDataset' object has no attribute 'get_run_splits'

Extra Trees outperforms RF when train and validation are mixed between runs, yet worse when asked to generalize to a new run. Given that the goal of this modelling is to generalize to unseen runs, I will proceed with RF for now.

### RF validation performance over n_estimators